In [4]:
!pip install tree-sitter
!pip install tree-sitter-java
!pip install tree-sitter-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.4/635.4 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 11.6 MB/s eta 0:00:00


In [5]:
import ast
import tree_sitter
from tree_sitter import Language, Parser
import tree_sitter_java
import tree_sitter_python

In [6]:
import pandas as pd
translation_pairs_df = pd.read_csv('/content/java_python_translation_pairs_corrected.csv')
display(translation_pairs_df.head())

,qid,text,java_code,python_code
0,602,Find the first repeated character in a given s...,import java.io.*;\nimport java.lang.*;\nimport...,"def first_repeated_char(str1):\n for index,c ..."
1,603,get a lucid number smaller than or equal to n.,import java.io.*;\nimport java.lang.*;\nimport...,def get_ludic(n):\n\tludics = []\n\tfor i in r...
2,604,reverse words in a given string.,import java.io.*;\nimport java.lang.*;\nimport...,def reverse_words(s):\n return ' '.join...
3,605,check if the given integer is a prime number.,import java.io.*;\nimport java.lang.*;\nimport...,def prime_num(num):\n if num >=1:\n for i i...
4,606,convert degrees to radians.,import java.io.*;\nimport java.lang.*;\nimport...,import math\ndef radian_degree(degree):\n radi...


In [7]:
JAVA_LANGUAGE = Language(
    tree_sitter_java.language()
)

parser = Parser()
parser.language = JAVA_LANGUAGE

In [8]:
def extract_java_ir(code):

    tree = parser.parse(
        bytes(code, "utf8")
    )

    root = tree.root_node

    ir = {
        "functions": 0,
        "for_loops": 0,
        "while_loops": 0,
        "ifs": 0,
        "returns": 0,
        "assignments": 0,
        "binary_ops": [],
        "comparisons": [],
        "calls": [],
        "data_structures": [],
        "recursion": False
    }

    method_names = set()

    def walk(node):

        if node.type == "method_declaration":

            ir["functions"] += 1

            for child in node.children:

                if child.type == "identifier":
                    method_names.add(
                        child.text.decode()
                    )

        elif node.type in [
            "for_statement",
            "enhanced_for_statement"
        ]:
            ir["for_loops"] += 1

        elif node.type == "while_statement":
            ir["while_loops"] += 1

        elif node.type == "if_statement":
            ir["ifs"] += 1

        elif node.type == "return_statement":
            ir["returns"] += 1

        elif node.type == "assignment_expression":
            ir["assignments"] += 1

        elif node.type == "method_invocation":

            for child in node.children:

                if child.type == "identifier":

                    name = child.text.decode()

                    ir["calls"].append(name)

                    if name in method_names:
                        ir["recursion"] = True

                    break

        elif node.type in [
            "+",
            "-",
            "*",
            "/",
            "%",
            "==",
            "!=",
            "<",
            ">",
            "<=",
            ">="
        ]:
            ir["binary_ops"].append(
                node.type
            )

        elif node.type == "array_creation_expression":
            ir["data_structures"].append(
                "array"
            )

        for child in node.children:
            walk(child)

    walk(root)

    for key in [
        "binary_ops",
        "comparisons",
        "calls",
        "data_structures"
    ]:
        ir[key] = sorted(
            list(set(ir[key]))
        )

    return ir

In [9]:
import ast

def extract_python_ir(code):

    tree = ast.parse(code)

    ir = {
        "functions": 0,
        "for_loops": 0,
        "while_loops": 0,
        "ifs": 0,
        "returns": 0,
        "assignments": 0,
        "binary_ops": [],
        "comparisons": [],
        "calls": [],
        "data_structures": [],
        "recursion": False
    }

    function_names = set()

    class Visitor(ast.NodeVisitor):

        def visit_FunctionDef(self, node):

            ir["functions"] += 1
            function_names.add(node.name)

            self.generic_visit(node)

        def visit_For(self, node):
            ir["for_loops"] += 1
            self.generic_visit(node)

        def visit_While(self, node):
            ir["while_loops"] += 1
            self.generic_visit(node)

        def visit_If(self, node):
            ir["ifs"] += 1
            self.generic_visit(node)

        def visit_Return(self, node):
            ir["returns"] += 1
            self.generic_visit(node)

        def visit_Assign(self, node):
            ir["assignments"] += 1
            self.generic_visit(node)

        def visit_Call(self, node):

            if isinstance(node.func, ast.Name):

                ir["calls"].append(node.func.id)

                if node.func.id in function_names:
                    ir["recursion"] = True

            self.generic_visit(node)

        def visit_BinOp(self, node):

            op = type(node.op).__name__

            ir["binary_ops"].append(op)

            self.generic_visit(node)

        def visit_Compare(self, node):

            for op in node.ops:
                ir["comparisons"].append(
                    type(op).__name__
                )

            self.generic_visit(node)

        def visit_List(self, node):
            ir["data_structures"].append("list")
            self.generic_visit(node)

        def visit_Dict(self, node):
            ir["data_structures"].append("dict")
            self.generic_visit(node)

        def visit_Set(self, node):
            ir["data_structures"].append("set")
            self.generic_visit(node)

        def visit_Tuple(self, node):
            ir["data_structures"].append("tuple")
            self.generic_visit(node)

    Visitor().visit(tree)

    for key in [
        "binary_ops",
        "comparisons",
        "calls",
        "data_structures"
    ]:
        ir[key] = sorted(list(set(ir[key])))

    return ir

In [10]:
def similarity(ir1, ir2):

    score = 0
    total = 0

    numeric_fields = [
        "functions",
        "for_loops",
        "while_loops",
        "ifs",
        "returns",
        "assignments"
    ]

    for field in numeric_fields:

        total += 1

        if ir1[field] == ir2[field]:
            score += 1

    list_fields = [
        "binary_ops",
        "comparisons",
        "calls",
        "data_structures"
    ]

    for field in list_fields:

        total += 1

        s1 = set(ir1[field])
        s2 = set(ir2[field])

        union = s1.union(s2)

        if len(union) == 0:
            score += 1
        else:
            score += (
                len(s1.intersection(s2))
                / len(union)
            )

    total += 1

    if ir1["recursion"] == ir2["recursion"]:
        score += 1

    return round(score / total, 4)

In [11]:
def compare_java_python(java_problem, python_code):
    java_ir = extract_java_ir(java_problem)
    python_ir = extract_python_ir(python_code)

    score = similarity(
        java_ir,
        python_ir
    )
    print(f"Similarity: {score}")
    return score

In [14]:
results_data = []
error_data = []

for index, row in translation_pairs_df.iterrows():
    java_code = row['java_code']
    python_code = row['python_code'] # Directly use the python_code from the DataFrame

    # Calculate similarity score
    try:
        score = compare_java_python(java_code, python_code) # Compare with the actual python_code
    except Exception as e:
        score = 0.0 # Assign a default score if an error occurs during comparison
        print(f"Error calculating similarity for qid {row['qid']}: {e}")
        error_data.append({
            'qid': row['qid'],
            'java_code': java_code,
            'python_code': python_code,
            'error_message': str(e)
        })

    # Collect all data for the new DataFrame
    results_data.append({
        'qid': row['qid'],
        'text': row['text'], # Include original 'text' column
        'java_code': java_code,
        'python_code': python_code, # Store the actual python_code used for comparison
        'ast_similarity_score': score
    })

results_df = pd.DataFrame(results_data)
error_df = pd.DataFrame(error_data)

print("Processing complete. Displaying the new DataFrame with results:")
display(results_df.head())

if not error_df.empty:
    print("\nDisplaying the DataFrame with errored records:")
    display(error_df.head())
else:
    print("\nNo errors were encountered during similarity calculation.")

Similarity: 0.6364
Similarity: 0.5455
Similarity: 0.6364
Similarity: 0.3636
Similarity: 0.7273
Similarity: 0.4545
Similarity: 0.5455
Similarity: 0.5455
Similarity: 0.6364
Similarity: 0.7273
Similarity: 0.7273
Similarity: 0.5455
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.4545
Similarity: 0.5455
Similarity: 0.5455
Similarity: 0.3636
Similarity: 0.7273
Similarity: 0.8182
Similarity: 0.6364
Similarity: 0.4545
Similarity: 0.2727
Similarity: 0.3636
Similarity: 0.5455
Similarity: 0.2727
Similarity: 0.6364
Similarity: 0.4545
Similarity: 0.7273
Similarity: 0.7273
Similarity: 0.5455
Similarity: 0.6364
Similarity: 0.3636
Similarity: 0.5455
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.6364
Similarity: 0.5455
Similarity: 0.7273
Similarity: 0.3636
Similarity: 0.7273
Similarity: 0.7273
Similarity: 0.4545
Similarity: 0.8182
Similarity: 0.5455
Similarity: 0.4848
Similarity: 0.4545
Similarity: 0.8182
Similarity: 0.4545
Similarity: 0.5455
Similarity: 0.1818
Similarity: 0.4545
Similarity: 

<unknown>:3: SyntaxWarning: invalid escape sequence '\B'
<unknown>:2: SyntaxWarning: invalid escape sequence '\.'
<unknown>:3: SyntaxWarning: invalid escape sequence '\W'
<unknown>:3: SyntaxWarning: invalid escape sequence '\W'
<unknown>:2: SyntaxWarning: invalid escape sequence '\.'
<unknown>:15: SyntaxWarning: invalid escape sequence '\s'
<unknown>:4: SyntaxWarning: invalid escape sequence '\A'
<unknown>:3: SyntaxWarning: invalid escape sequence '\d'
<unknown>:3: SyntaxWarning: invalid escape sequence '\d'


,qid,text,java_code,python_code,ast_similarity_score
0,602,Find the first repeated character in a given s...,import java.io.*;\nimport java.lang.*;\nimport...,"def first_repeated_char(str1):\n for index,c ...",0.6364
1,603,get a lucid number smaller than or equal to n.,import java.io.*;\nimport java.lang.*;\nimport...,def get_ludic(n):\n\tludics = []\n\tfor i in r...,0.5455
2,604,reverse words in a given string.,import java.io.*;\nimport java.lang.*;\nimport...,def reverse_words(s):\n return ' '.join...,0.6364
3,605,check if the given integer is a prime number.,import java.io.*;\nimport java.lang.*;\nimport...,def prime_num(num):\n if num >=1:\n for i i...,0.3636
4,606,convert degrees to radians.,import java.io.*;\nimport java.lang.*;\nimport...,import math\ndef radian_degree(degree):\n radi...,0.7273



No errors were encountered during similarity calculation.


In [15]:
mean_similarity_score = results_df['ast_similarity_score'].mean()
print(f"Mean AST Similarity Score: {mean_similarity_score:.4f}")

Mean AST Similarity Score: 0.5424
